# 04c Patience check
Additional check, not required by the project: did patience 3 stop the chosen model 2 too early? Model 1 uses patience 10 and model 2 patience 3, because one epoch of model 2 takes minutes (`04_model2_rnn.ipynb`, section "Training"). The chosen configuration of model 2 (GRU, RNN hidden size 128, learning rate 0.001, dropout 0.5, batch size 32, seed 42) is trained once more with patience 10 instead of 3 and at most 30 epochs instead of 15 (the chosen run stopped at epoch 7, so it did not reach its limit); everything else stays the same.

**Fixed before the run:** this is a sensitivity check. The chosen model 2 stays the run with patience 3, whatever the result. If the validation loss gets lower after epoch 7, this is reported as patience 3 having stopped this configuration too early. Only this configuration is repeated, so all other runs keep patience 3, and the result does not show the best patience for other configurations. The test set is not loaded.

## Data, model and evaluation
The same input, classes, batches and `evaluate()` as in `04_model2_rnn.ipynb`.

In [1]:
import numpy as np
import torch

data = np.load("features/model2.npz")
# token indices of O1, O2, h1, h2 for every example, and the labels (0 = h1, 1 = h2)
X_train = torch.tensor(data["X_train"])
y_train = torch.tensor(data["y_train"])
X_valid = torch.tensor(data["X_valid"])
y_valid = torch.tensor(data["y_valid"])
# GloVe vectors: row i = vector of the word with index i (row 0 = padding)
embedding_matrix = torch.tensor(data["embedding_matrix"])
print(X_train.shape, X_valid.shape, embedding_matrix.shape)

torch.Size([16801, 4, 34]) torch.Size([1000, 4, 34]) torch.Size([15465, 300])


In [2]:
# same classes as in 04_model2_rnn.ipynb
import torch.nn as nn

# same class as in 03_model1_embeddings.ipynb
class Classifier(nn.Module):

    def __init__(self, input_dim, output_dim, inner_dim=10):
        super().__init__()
        self.linear1 = nn.Linear(input_dim, inner_dim)
        self.relu = nn.ReLU()
        self.linear2 = nn.Linear(inner_dim, output_dim)

    def forward(self, x):
        out = self.linear1(x)
        out = self.relu(out)
        out = self.linear2(out)
        return out

from torch.nn.utils.rnn import pack_padded_sequence

class RNNClassifier(nn.Module):

    def __init__(self, embedding_matrix, rnn_type="lstm", hidden_dim=128, num_layers=2, dropout=0.0, inner_dim=100):
        super().__init__()
        # embedding layer: the frozen GloVe vectors, row 0 = padding
        self.embedding = nn.Embedding.from_pretrained(embedding_matrix, freeze=True, padding_idx=0)
        # size of one word vector (300)
        embedding_dim = embedding_matrix.shape[1]
        # 2-layer LSTM or GRU; batch_first=True: input shape [batch size, sequence length, embedding dim]
        if rnn_type == "lstm":
            self.rnn = nn.LSTM(embedding_dim, hidden_dim, num_layers=num_layers, dropout=dropout, batch_first=True)
        else:
            self.rnn = nn.GRU(embedding_dim, hidden_dim, num_layers=num_layers, dropout=dropout, batch_first=True)
        # dropout on the 4 final hidden states before the classifier (0.0 = no dropout)
        self.dropout = nn.Dropout(dropout)
        # classifier on the 4 final hidden states side by side (O1, O2, h1, h2)
        self.classifier = Classifier(4 * hidden_dim, 2, inner_dim=inner_dim)

    def forward(self, x):
        # x: token indices, shape [batch size, 4, 34] (O1, O2, h1, h2)
        features = []
        for i in range(4):
            # one text (O1, O2, h1 or h2) of every example in the batch: [batch size, 34]
            text = x[:, i]
            # real length of each text = number of token indices that are not 0 (padding)
            lengths = (text != 0).sum(dim=1)
            # look up the GloVe vectors: [batch size, 34, 300]
            embedded = self.embedding(text)
            # give the RNN the real lengths, so it stops after the last real word
            packed = pack_padded_sequence(embedded, lengths.cpu(), batch_first=True, enforce_sorted=False)
            # the RNN reads the text; only the final hidden state is needed
            _, hidden = self.rnn(packed)
            # the LSTM returns (h_n, c_n), the GRU only h_n
            if isinstance(self.rnn, nn.LSTM):
                hidden = hidden[0]
            # h_n of the last layer: [batch size, hidden dim]
            features.append(hidden[-1])
        # the 4 final hidden states side by side: [batch size, 4 * hidden dim]
        features = torch.concat(features, dim=-1)
        # dropout on the final hidden states (active only in training mode)
        features = self.dropout(features)
        return self.classifier(features)

In [3]:
from torch.optim import Adam
from torch.utils.data import TensorDataset, DataLoader

criterion = nn.CrossEntropyLoss()

# the same batches as in 04_model2_rnn.ipynb
train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=32, shuffle=True, drop_last=True)
train_eval_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=317)
valid_loader = DataLoader(TensorDataset(X_valid, y_valid), batch_size=100)

# same function as in 04_model2_rnn.ipynb
def evaluate(model, loader):
    # evaluation mode, and no gradients are needed
    model.eval()
    losses, predictions, labels = [], [], []
    with torch.no_grad():
        for x, y in loader:
            output = model(x)
            # loss of this batch
            losses.append(criterion(output, y).item())
            # predicted class: the hypothesis with the higher logit
            predictions.append(output.argmax(dim=1))
            labels.append(y)
    # join the batches: one prediction and one label per example
    predictions = torch.cat(predictions)
    labels = torch.cat(labels)
    # share of correct predictions
    accuracy = (predictions == labels).float().mean().item()
    return np.mean(losses), accuracy

## Training
The same training as `train()` in `04_model2_rnn.ipynb`, with patience 10 and at most 30 epochs. The run has its own name and checkpoint (`-p10`), so the chosen checkpoint is not overwritten. It is logged to W&B with the model name `glove-gru-patience-check`, so the saved view of the candidate runs does not show it.

In [4]:
import os
import time

# no W&B log lines in the notebook output; the metrics are still logged
os.environ["WANDB_SILENT"] = "true"

import wandb
wandb.login()

os.makedirs("checkpoints", exist_ok=True)

# the chosen configuration of model 2, with patience 10
rnn_type, hidden_dim, learning_rate, dropout, max_epochs, patience = "gru", 128, 0.001, 0.5, 30, 10
name = f"glove-{rnn_type}-h{hidden_dim}-lr{learning_rate}-d{dropout}-p{patience}"
checkpoint_path = f"checkpoints/{name}.pth"
wandb.init(
    project="nlp-project1-art-final",
    name=name,
    config={
        "model": "glove-gru-patience-check",
        "rnn_type": rnn_type,
        "hidden_dim": hidden_dim,
        "num_layers": 2,
        "inner_dim": 100,
        "dropout": dropout,
        "learning_rate": learning_rate,
        "max_epochs": max_epochs,
        "patience": patience,
        "batch_size": train_loader.batch_size,
        "seed": 42,
    },
)

# same start weights and batches as the chosen run
torch.manual_seed(42)
model = RNNClassifier(embedding_matrix, rnn_type=rnn_type, hidden_dim=hidden_dim, dropout=dropout)
optimizer = Adam(model.parameters(), lr=learning_rate)
best_valid_loss = float("inf")
# number of epochs in a row without a new lowest validation loss
epochs_without_improvement = 0
# validation loss of every epoch, for the comparison below
valid_losses = []

for epoch in range(max_epochs):
    start = time.time()
    model.train()
    for x, y in train_loader:
        output = model(x)
        loss = criterion(output, y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    train_loss, train_acc = evaluate(model, train_eval_loader)
    valid_loss, valid_acc = evaluate(model, valid_loader)
    valid_losses.append(valid_loss)
    minutes = (time.time() - start) / 60
    print(f"Epoch {epoch + 1}: train loss {train_loss:.4f}, train acc {train_acc:.2%}, valid loss {valid_loss:.4f}, valid acc {valid_acc:.2%} ({minutes:.1f} min)")
    wandb.log({"epoch": epoch + 1, "train_loss": train_loss, "train_acc": train_acc, "val_loss": valid_loss, "val_acc": valid_acc})
    if valid_loss < best_valid_loss:
        best_valid_loss = valid_loss
        best_valid_acc = valid_acc
        best_epoch = epoch + 1
        torch.save(model.state_dict(), checkpoint_path)
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1
        if epochs_without_improvement == patience:
            print(f"Early stopping: no lower validation loss in the last {patience} epochs")
            break

wandb.run.summary["best_val_loss"] = best_valid_loss
wandb.run.summary["best_val_acc"] = best_valid_acc
wandb.run.summary["best_epoch"] = best_epoch
wandb.finish()
print(f"Best epoch: {best_epoch}, valid loss {best_valid_loss:.4f}, valid acc {best_valid_acc:.2%}")

Epoch 1: train loss 0.6130, train acc 65.74%, valid loss 0.6349, valid acc 64.10% (0.8 min)
Epoch 2: train loss 0.5739, train acc 69.08%, valid loss 0.6051, valid acc 66.80% (0.9 min)
Epoch 3: train loss 0.5316, train acc 72.76%, valid loss 0.5942, valid acc 66.20% (1.0 min)
Epoch 4: train loss 0.4934, train acc 76.77%, valid loss 0.5872, valid acc 67.60% (0.9 min)
Epoch 5: train loss 0.4157, train acc 81.17%, valid loss 0.6144, valid acc 68.10% (1.0 min)
Epoch 6: train loss 0.3357, train acc 85.97%, valid loss 0.6449, valid acc 66.20% (1.0 min)
Epoch 7: train loss 0.2674, train acc 90.28%, valid loss 0.6926, valid acc 66.50% (1.1 min)
Epoch 8: train loss 0.1774, train acc 93.55%, valid loss 0.8698, valid acc 64.60% (1.0 min)
Epoch 9: train loss 0.1423, train acc 95.43%, valid loss 0.8957, valid acc 65.90% (1.0 min)
Epoch 10: train loss 0.0976, train acc 97.39%, valid loss 0.9716, valid acc 63.70% (1.0 min)
Epoch 11: train loss 0.0662, train acc 98.24%, valid loss 1.1361, valid acc 66.

## Comparison with the chosen run
The seed and all settings are the same, so the first 7 epochs must be the same as in the chosen run (`04_model2_rnn.ipynb`, section "Dropout"). If they differ, something other than the patience changed, and the result cannot be attributed to the patience. If the best epoch is still epoch 4, the weights of the two best checkpoints must also be the same.

In [5]:
# validation loss per epoch of the chosen run, printed in 04_model2_rnn.ipynb
chosen_losses = [0.6349, 0.6051, 0.5942, 0.5872, 0.6144, 0.6449, 0.6926]
print("First 7 epochs the same as in the chosen run:", [round(float(value), 4) for value in valid_losses[:7]] == chosen_losses)

print("Patience 3 (chosen run): best epoch 4 of 7, valid loss 0.5872, valid acc 67.60%")
print(f"Patience 10: best epoch {best_epoch} of {len(valid_losses)}, valid loss {best_valid_loss:.4f}, valid acc {best_valid_acc:.2%}")

# the same best checkpoint?
chosen_weights = torch.load("checkpoints/glove-gru-h128-lr0.001-d0.5.pth", weights_only=True)
new_weights = torch.load(checkpoint_path, weights_only=True)
print("Same weights as the chosen checkpoint:", all(torch.equal(chosen_weights[key], new_weights[key]) for key in chosen_weights))

First 7 epochs the same as in the chosen run: True
Patience 3 (chosen run): best epoch 4 of 7, valid loss 0.5872, valid acc 67.60%
Patience 10: best epoch 4 of 14, valid loss 0.5872, valid acc 67.60%
Same weights as the chosen checkpoint: True


## Result
With patience 10, the run stops at epoch 14 and the best epoch stays epoch 4 (validation loss 0.5872, accuracy 67.60%). The first 7 epochs are the same as in the chosen run, and the best checkpoint has the same weights, so no other change affects this comparison. In the 10 epochs after epoch 4, the validation loss never gets lower: it rises to 1.3740 at epoch 14, while the training accuracy reaches 99.33%. So for this configuration and seed, patience 3 did not stop the training too early. This does not show the best patience for other configurations. The chosen model 2 stays, as fixed before the run.

## Sources
Course materials: NLP HS26 by A. Marfurt, Lucerne University of Applied Sciences and Arts (HSLU), 2026. Full references in README.md.
- Data, classes, batches, `evaluate()`, training and W&B: the same as in `04_model2_rnn.ipynb` (see the sources there).
- Early stopping with a patience: course slides "Project Discussions" (Week 6), slide 38.
- Number of training runs depending on the budget: course slides "Project Discussions" (Week 7), slide 55.
- Comparison: `torch.equal`, PyTorch documentation, https://docs.pytorch.org/docs/stable/generated/torch.equal.html; `round()`, `float()` and `all()`, Python documentation, https://docs.python.org/3/library/functions.html.
- Tools: Claude (Anthropic), ChatGPT (OpenAI) and Gemini (Google), see "Use of AI" in README.md.